In [1]:
import ast
import json
import random
import re
import time

import pandas as pd

from bs4 import BeautifulSoup
from curl_cffi import requests
from pathlib import Path
from typing import Any

from dataclasses import dataclass
from enum import Enum


In [2]:
from dataclasses import dataclass
from enum import Enum


@dataclass(frozen=True)
class WhoScoredLeagueConfig:
    """Stores the static WhoScored configuration for a competition."""

    country_name: str
    league_name: str
    region_id: int
    tournament_id: int
    slug: str
    season: str

    @property
    def url(self) -> str:
        """Builds the WhoScored competition URL."""
        return (
            f"https://www.whoscored.com/regions/"
            f"{self.region_id}/tournaments/{self.tournament_id}/{self.slug}"
        )


class WhoScoredLeague(Enum):
    """Available competitions supported by the scraper."""

    # =========================
    # Top European leagues
    # =========================

    ENGLAND_PREMIER_LEAGUE = WhoScoredLeagueConfig(
        country_name="England",
        league_name="Premier League",
        region_id=252,
        tournament_id=2,
        slug="england-premier-league",
        season="2025_2026",
    )

    SPAIN_LALIGA = WhoScoredLeagueConfig(
        country_name="Spain",
        league_name="LaLiga",
        region_id=206,
        tournament_id=4,
        slug="spain-laliga",
        season="2025_2026",
    )

    ITALY_SERIE_A = WhoScoredLeagueConfig(
        country_name="Italy",
        league_name="Serie A",
        region_id=108,
        tournament_id=5,
        slug="italy-serie-a",
        season="2025_2026",
    )

    GERMANY_BUNDESLIGA = WhoScoredLeagueConfig(
        country_name="Germany",
        league_name="Bundesliga",
        region_id=81,
        tournament_id=3,
        slug="germany-bundesliga",
        season="2025_2026",
    )

    FRANCE_LIGUE_1 = WhoScoredLeagueConfig(
        country_name="France",
        league_name="Ligue 1",
        region_id=74,
        tournament_id=22,
        slug="france-ligue-1",
        season="2025_2026",
    )

    PORTUGAL_LIGA_PORTUGAL = WhoScoredLeagueConfig(
        country_name="Portugal",
        league_name="Liga Portugal",
        region_id=177,
        tournament_id=21,
        slug="portugal-liga-portugal",
        season="2025_2026",
    )

    NETHERLANDS_EREDIVISIE = WhoScoredLeagueConfig(
        country_name="Netherlands",
        league_name="Eredivisie",
        region_id=155,
        tournament_id=13,
        slug="netherlands-eredivisie",
        season="2025_2026",
    )

    BELGIUM_JUPILER_PRO_LEAGUE = WhoScoredLeagueConfig(
        country_name="Belgium",
        league_name="Jupiler Pro League",
        region_id=22,
        tournament_id=18,
        slug="belgium-jupiler-pro-league",
        season="2025_2026",
    )

    SCOTLAND_PREMIERSHIP = WhoScoredLeagueConfig(
        country_name="Scotland",
        league_name="Premiership",
        region_id=253,
        tournament_id=20,
        slug="scotland-premiership",
        season="2025_2026",
    )

    TURKEY_SUPER_LIG = WhoScoredLeagueConfig(
        country_name="Turkey",
        league_name="Super Lig",
        region_id=225,
        tournament_id=17,
        slug="turkey-super-lig",
        season="2025_2026",
    )

    RUSSIA_PREMIER_LEAGUE = WhoScoredLeagueConfig(
        country_name="Russia",
        league_name="Premier League",
        region_id=182,
        tournament_id=77,
        slug="russia-premier-league",
        season="2025_2026",
    )

    # =========================
    # England domestic competitions
    # =========================

    ENGLAND_CHAMPIONSHIP = WhoScoredLeagueConfig(
        country_name="England",
        league_name="Championship",
        region_id=252,
        tournament_id=7,
        slug="england-championship",
        season="2025_2026",
    )

    ENGLAND_LEAGUE_ONE = WhoScoredLeagueConfig(
        country_name="England",
        league_name="League One",
        region_id=252,
        tournament_id=8,
        slug="england-league-one",
        season="2025_2026",
    )

    ENGLAND_LEAGUE_TWO = WhoScoredLeagueConfig(
        country_name="England",
        league_name="League Two",
        region_id=252,
        tournament_id=9,
        slug="england-league-two",
        season="2025_2026",
    )

    ENGLAND_FA_CUP = WhoScoredLeagueConfig(
        country_name="England",
        league_name="FA Cup",
        region_id=252,
        tournament_id=26,
        slug="england-fa-cup",
        season="2025_2026",
    )

    ENGLAND_LEAGUE_CUP = WhoScoredLeagueConfig(
        country_name="England",
        league_name="League Cup",
        region_id=252,
        tournament_id=29,
        slug="england-league-cup",
        season="2025_2026",
    )

    ENGLAND_WOMENS_SUPER_LEAGUE = WhoScoredLeagueConfig(
        country_name="England",
        league_name="Women's Super League",
        region_id=252,
        tournament_id=739,
        slug="england-women-s-super-league",
        season="2025_2026",
    )

    # =========================
    # Other domestic leagues
    # =========================

    BRAZIL_BRASILEIRAO = WhoScoredLeagueConfig(
        country_name="Brazil",
        league_name="Brasileirão",
        region_id=31,
        tournament_id=95,
        slug="brazil-brasileirao",
        season="2026",
    )

    GERMANY_2_BUNDESLIGA = WhoScoredLeagueConfig(
        country_name="Germany",
        league_name="2. Bundesliga",
        region_id=81,
        tournament_id=6,
        slug="germany-2-bundesliga",
        season="2025_2026",
    )

    USA_MAJOR_LEAGUE_SOCCER = WhoScoredLeagueConfig(
        country_name="USA",
        league_name="Major League Soccer",
        region_id=233,
        tournament_id=85,
        slug="usa-major-league-soccer",
        season="2026",
    )

    # =========================
    # Continental / international competitions
    # =========================

    EUROPE_CHAMPIONS_LEAGUE = WhoScoredLeagueConfig(
        country_name="Europe",
        league_name="Champions League",
        region_id=250,
        tournament_id=12,
        slug="europe-champions-league",
        season="2025_2026",
    )

    EUROPE_EUROPA_LEAGUE = WhoScoredLeagueConfig(
        country_name="Europe",
        league_name="Europa League",
        region_id=250,
        tournament_id=30,
        slug="europe-europa-league",
        season="2025_2026",
    )

    INTERNATIONAL_FIFA_CLUB_WORLD_CUP = WhoScoredLeagueConfig(
        country_name="International",
        league_name="FIFA Club World Cup",
        region_id=247,
        tournament_id=67,
        slug="international-fifa-club-world-cup",
        season="2025",
    )

    INTERNATIONAL_WORLD_CUP_QUALIFICATION_UEFA = WhoScoredLeagueConfig(
        country_name="International",
        league_name="World Cup Qualification UEFA",
        region_id=247,
        tournament_id=721,
        slug="international-world-cup-qualification-uefa",
        season="2025_2026",
    )

    INTERNATIONAL_UEFA_NATIONS_LEAGUE_A = WhoScoredLeagueConfig(
        country_name="International",
        league_name="UEFA Nations League A",
        region_id=247,
        tournament_id=683,
        slug="international-uefa-nations-league-a",
        season="2025_2026",
    )

In [3]:
# Selected WhoScored competition configuration.
WHOSCORED_CONFIG = WhoScoredLeague.NETHERLANDS_EREDIVISIE.value


def build_match_events_output_path(
    whoscored_config: WhoScoredLeagueConfig,
) -> str:
    """Builds the JSON output path for match events."""
    safe_country = whoscored_config.country_name.lower().replace(" ", "_")
    safe_league = whoscored_config.league_name.lower().replace(" ", "_")
    safe_season = whoscored_config.season.replace("/", "_")

    return f"data/match_events/{safe_country}_{safe_league}_{safe_season}.json"


# Output file for scraped match events.
MATCH_EVENTS_JSON_PATH = build_match_events_output_path(
    whoscored_config=WHOSCORED_CONFIG,
)

# Input file containing stored fixture data.
FIXTURES_JSON_PATH = "data/football_fixtures.json"

In [4]:
def to_slug(value: str) -> str:
    """Converts a value into a WhoScored-style URL slug."""
    return "-".join(str(value).lower().split())


def _fetch_html(
    url: str,
    retries: int = 3,
    timeout: int = 30,
) -> str:
    """Fetches a page HTML with retries and browser impersonation."""
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(
                url,
                impersonate="chrome120",
                timeout=timeout,
            )

            response.raise_for_status()
            return response.text

        except Exception as exc:
            if attempt == retries:
                raise

            # Backoff between retries to avoid retrying too aggressively.
            sleep_time = random.uniform(5, 12) * attempt
            print(
                f"Request failed. Retry {attempt}/{retries} "
                f"after {sleep_time:.2f}s. Error: {exc}"
            )
            time.sleep(sleep_time)

    raise RuntimeError("Unexpected fetch failure")

In [5]:
def get_teams_by_league_from_whoscored(league_url: str) -> list[dict[str, str]]:
    """Extracts team IDs and names from a WhoScored league page."""
    html = _fetch_html(url=league_url)

    soup = BeautifulSoup(html, "html.parser")

    # Find the script containing the league standings data.
    script_tag = None
    for script in soup.find_all("script"):
        if script.string and "standings" in script.string:
            script_tag = script

    if not script_tag:
        raise Exception("standings script not found")

    raw_standings = _extract_league_standings_arrays(
        script_text=script_tag.string,
        key="standings",
    )

    # Some leagues split standings into multiple arrays/groups.
    if len(raw_standings) > 1:
        standings = ast.literal_eval(raw_standings[0]) + ast.literal_eval(raw_standings[1])
    else:
        standings = ast.literal_eval(raw_standings[0])

    print("Total teams:", len(standings))

    teams = []
    for row in standings:
        teams.append({row[1]: row[2]})

    return teams


def _extract_league_standings_arrays(script_text: str, key: str) -> list[str]:
    """Extracts standings arrays from the league page script."""
    marker = f'"{key}":'
    start_search = 0
    extracted_arrays = []

    while True:
        marker_index = script_text.find(marker, start_search)
        if marker_index == -1:
            break

        text_after_key = script_text[marker_index + len(marker):]

        array_start = text_after_key.find("[")
        if array_start == -1:
            start_search = marker_index + len(marker)
            continue

        # Track nested brackets to capture the full array.
        bracket_count = 0
        array_end = None

        for index, char in enumerate(text_after_key[array_start:]):
            if char == "[":
                bracket_count += 1
            elif char == "]":
                bracket_count -= 1

                if bracket_count == 0:
                    array_end = array_start + index + 1
                    break

        if array_end is None:
            raise ValueError(f'Could not find array end for "{key}"')

        extracted_arrays.append(text_after_key[array_start:array_end])

        start_search = marker_index + len(marker) + array_end

    if not extracted_arrays:
        raise ValueError(f'Could not find "{key}"')

    return extracted_arrays

In [6]:
def build_whoscored_fixture_urls(
    league_teams: list[dict[int, str]],
    country_name: str,
) -> list[str]:
    """Builds WhoScored fixture URLs for all teams in a league."""
    urls: list[str] = []

    for team in league_teams:
        for team_id, club_name in team.items():
            formatted_team_name = to_slug(club_name)

            url = (
                f"https://www.whoscored.com/teams/"
                f"{team_id}/fixtures/{country_name}-{formatted_team_name}"
            )

            urls.append(url)

    return urls


def get_team_fixtures_from_whoscored(team_url: str) -> list[dict[str, Any]]:
    """Extracts fixture data from a WhoScored team fixtures page."""
    html = _fetch_html(url=team_url)

    soup = BeautifulSoup(html, "html.parser")

    # Find the script containing fixture data.
    fixtures_script = None
    for script in soup.find_all("script"):
        if script.string and "fixtureMatches" in script.string:
            fixtures_script = script
            break

    if not fixtures_script:
        raise Exception("fixtureMatches script not found")

    raw_fixtures = fixtures_script.text.split("fixtureMatches: ")[1]
    raw_fixtures = raw_fixtures.split("};")[0].strip()

    # Replace empty JS array values so Python can parse them.
    while ",," in raw_fixtures:
        raw_fixtures = raw_fixtures.replace(",,", ",None,")

    raw_fixtures = raw_fixtures.replace("[,", "[None,")
    raw_fixtures = raw_fixtures.replace(",]", ",None]")

    fixture_rows = ast.literal_eval(raw_fixtures)

    fixture_columns = [
        "match_id", "status_code", "date", "time",
        "home_team_id", "home_team", "home_flag",
        "away_team_id", "away_team", "away_flag",
        "score", "ht_score", "started", "cancelled", "status",
        "season", "competition", "winner",
        "competition_id", "region_id", "tournament_id", "stage_id",
        "competition_code", "home_country_code", "away_country_code",
        "neutral_venue", "flag_1", "flag_2",
        "home_country", "away_country", "competition_country",
        "home_goals", "away_goals",
    ]

    fixtures = [dict(zip(fixture_columns, row)) for row in fixture_rows]

    # Build the direct match URL for each fixture.
    for fixture in fixtures:
        match_id = fixture.get("match_id")
        competition_country_slug = to_slug(fixture.get("competition_country", ""))
        competition_slug = to_slug(fixture.get("competition", ""))
        season = str(fixture.get("season", ""))
        home_team_slug = to_slug(fixture.get("home_team", ""))
        away_team_slug = to_slug(fixture.get("away_team", ""))

        match_slug = (
            f"{competition_country_slug}-{competition_slug}-"
            f"{home_team_slug}-{away_team_slug}"
        )

        fixture["match_url"] = (
            f"https://www.whoscored.com/matches/{match_id}/live/{match_slug}"
        )

    return fixtures


def scrape_league_fixtures(
    league_fixture_urls: list[str],
    min_sleep: float = 4.0,
    max_sleep: float = 6.0,
) -> list[dict[str, Any]]:
    """Scrapes fixtures from all team pages in a league."""
    all_fixtures: list[dict[str, Any]] = []

    for index, team_url in enumerate(league_fixture_urls, start=1):
        print(f"[{index}/{len(league_fixture_urls)}] Scraping: {team_url}")

        try:
            fixtures = get_team_fixtures_from_whoscored(team_url)
            all_fixtures.extend(fixtures)

        except Exception as exc:
            print(f"Failed to scrape {team_url}: {exc}")

        # Small random delay to avoid scraping too aggressively.
        sleep_time = random.uniform(min_sleep, max_sleep)
        print(f"Sleeping for {sleep_time:.2f}s")
        time.sleep(sleep_time)

    return _deduplicate_fixtures_by_match_id(fixtures=all_fixtures)


def _deduplicate_fixtures_by_match_id(
    fixtures: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Removes duplicated fixtures using the WhoScored match ID."""
    unique_fixtures: dict[Any, dict[str, Any]] = {}

    for fixture in fixtures:
        match_id = fixture.get("match_id")

        if match_id is None:
            continue

        unique_fixtures[match_id] = fixture

    return list(unique_fixtures.values())

In [7]:
def save_fixtures_to_json(
    fixtures: list[dict[str, Any]],
    country_name: str,
    league_name: str,
    season: str,
    output_path: str | Path,
) -> dict[str, Any]:
    """Stores league fixtures in a nested JSON structure."""
    output_path = Path(output_path)

    if output_path.exists():
        with output_path.open("r", encoding="utf-8") as file:
            data: dict[str, Any] = json.load(file)
    else:
        data = {}

    data.setdefault(country_name, {})
    data[country_name].setdefault(league_name, {})
    data[country_name][league_name][season] = fixtures

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with output_path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)

    print(
        f"Successfully stored: {len(fixtures)} fixtures | "
        f"{country_name} - {league_name}"
    )

    return data


def extract_match_urls(
    fixtures: list[dict[str, Any]],
    country_name: str,
    league_name: str,
) -> dict[str, list[str]]:
    """Splits fixture match URLs into played and not played groups."""
    filter_competition = f"{to_slug(country_name)}-{to_slug(league_name)}"

    result: dict[str, list[str]] = {
        "fixtures_played": [],
        "fixtures_not_played": [],
    }

    for fixture in fixtures:
        match_url = fixture.get("match_url")

        if not match_url:
            continue

        if filter_competition not in str(match_url):
            continue

        if fixture.get("status") is None:
            result["fixtures_not_played"].append(str(match_url))
        else:
            result["fixtures_played"].append(str(match_url))

    return result

In [8]:
def get_match_events_from_whoscored(match_url: str) -> dict[str, Any] | None:
    """Extracts match events and player names from a WhoScored match page."""
    html = _fetch_html(url=match_url)

    soup = BeautifulSoup(html, "html.parser")

    # Find the script containing match event data.
    script_tag = None
    for script in soup.find_all("script"):
        if script.string and "matchCentreData" in script.string:
            script_tag = script
            break

    if not script_tag or not script_tag.string:
        raise Exception("matchCentreData script not found")

    # Extract the matchCentreData object from the page script.
    match_obj = re.search(
        r"matchCentreData\s*:\s*({.*?})\s*,\s*matchCentreEventTypeJson",
        script_tag.string,
        re.DOTALL,
    )

    if not match_obj:
        raise Exception("matchCentreData object not found")

    print("Successfully parsed matchCentreData")

    match_json = json.loads(match_obj.group(1))

    # Player dictionary used later to enrich events with names.
    player_id_name_dict = match_json.get("playerIdNameDictionary", {})

    if not player_id_name_dict:
        print("Warning: No player ID-name dictionary found")

    events_dict = match_json.get("events", [])

    if not events_dict:
        print("No events data found in matchCentreData")
        return None

    return {
        "events": events_dict,
        "players_id_name": player_id_name_dict,
    }

def _extract_match_id_from_url(match_url: str) -> str:
    """Extracts the WhoScored match ID from a match URL."""
    parts = match_url.split("/")

    try:
        match_index = parts.index("matches")
        return parts[match_index + 1]
    except ValueError:
        return ""


def scrape_single_match_events(match_url: str) -> dict[str, Any]:
    """Scrapes one match and enriches events with player names."""
    match_id = _extract_match_id_from_url(match_url=match_url)

    try:
        match_events = get_match_events_from_whoscored(match_url=match_url)

        if match_events is None:
            raise ValueError("No match events found")

        events_dict = match_events["events"]
        players_in_events = match_events["players_id_name"]

        # Normalize player IDs as strings to avoid int/string mismatches.
        players_in_events = {
            str(player_id): player_name
            for player_id, player_name in players_in_events.items()
        }

        # Add player names when player IDs are available.
        for event in events_dict:
            player_id = event.get("playerId")
            related_player_id = event.get("relatedPlayerId")

            if player_id is not None:
                event["playerName"] = players_in_events.get(str(player_id))

            if related_player_id is not None:
                event["relatedPlayerName"] = players_in_events.get(str(related_player_id))

        return {
            "match_id": match_id,
            "match_url": match_url,
            "status": "success",
            "events_count": len(events_dict),
            "events": events_dict,
            "error": None,
        }

    except Exception as exc:
        return {
            "match_id": match_id,
            "match_url": match_url,
            "status": "failed",
            "events_count": 0,
            "events": [],
            "error": str(exc),
        }
    
def _load_json_file(path: str | Path) -> dict[str, Any]:
    """Loads a JSON file or returns an empty dict if it does not exist."""
    path = Path(path)

    if not path.exists():
        return {}

    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def _save_json_file(path: str | Path, data: dict[str, Any]) -> None:
    """Saves data to a JSON file, creating parent folders when needed."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        json.dump(data, file, indent=4, ensure_ascii=False)


def _chunk_list(items: list[str], chunk_size: int) -> list[list[str]]:
    """Splits a list into smaller batches."""
    return [
        items[index:index + chunk_size]
        for index in range(0, len(items), chunk_size)
    ]


def scrape_match_events_in_batches(
    match_urls: list[str],
    country_name: str,
    league_name: str,
    season: str,
    output_path: str | Path,
    batch_size: int = 10,
    min_sleep_between_matches=6.0,
    max_sleep_between_matches=15.0,
    min_sleep_between_batches=90.0,
    max_sleep_between_batches=240.0,
) -> dict[str, Any]:
    """Scrapes match events in batches and stores progress after each match."""
    data = _load_json_file(output_path)

    data.setdefault(country_name, {})
    data[country_name].setdefault(league_name, {})
    data[country_name][league_name].setdefault(season, {})

    season_data = data[country_name][league_name][season]
    season_data.setdefault("matches", {})

    batches = _chunk_list(match_urls, batch_size)

    for batch_index, batch in enumerate(batches, start=1):
        print(f"Starting batch {batch_index}/{len(batches)}")

        for match_index, match_url in enumerate(batch, start=1):
            match_id = _extract_match_id_from_url(match_url)

            # Skip matches that were already scraped successfully.
            if match_id in season_data["matches"]:
                existing_status = season_data["matches"][match_id].get("status")

                if existing_status == "success":
                    print(f"Skipping already scraped match: {match_id}")
                    continue

            print(f"Scraping match {match_index}/{len(batch)}: {match_url}")

            match_result = scrape_single_match_events(match_url)

            season_data["matches"][match_id] = match_result

            # Save after each match to avoid losing progress.
            _save_json_file(output_path, data)

            sleep_time = random.uniform(
                min_sleep_between_matches,
                max_sleep_between_matches,
            )

            print(f"Sleeping {sleep_time:.2f}s between matches")
            time.sleep(sleep_time)

        # Add a longer pause between batches.
        if batch_index < len(batches):
            batch_sleep = random.uniform(
                min_sleep_between_batches,
                max_sleep_between_batches,
            )

            print(f"Batch {batch_index} finished. Sleeping {batch_sleep:.2f}s")
            time.sleep(batch_sleep)

    return data

In [9]:
def load_fixtures_from_json(
    country_name: str,
    league_name: str,
    season: str,
    file_path: str | Path = "data/fixtures.json",
) -> list[dict[str, Any]]:
    """Loads stored fixtures for a specific country, league, and season."""
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"Fixtures file not found: {file_path}")

    with file_path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    try:
        fixtures = data[country_name][league_name][season]
    except KeyError:
        return []

    if not isinstance(fixtures, list):
        raise ValueError(
            f"Expected fixtures to be a list for "
            f"{country_name} -> {league_name} -> {season}"
        )

    return fixtures

In [11]:
def load_match_events_from_json(
    country_name: str,
    league_name: str,
    season: str,
    file_path: str | Path,
) -> dict[str, Any]:
    """Loads stored match events for a specific country, league, and season."""
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"Match events file not found: {file_path}")

    with file_path.open("r", encoding="utf-8") as file:
        data = json.load(file)

    try:
        season_data = data[country_name][league_name][season]
    except KeyError:
        return {}

    return season_data


def load_all_match_events_df(
    country_name: str,
    league_name: str,
    season: str,
    file_path: str | Path,
) -> pd.DataFrame:
    """Loads all successful match events and returns them as a DataFrame."""
    season_data = load_match_events_from_json(
        country_name=country_name,
        league_name=league_name,
        season=season,
        file_path=file_path,
    )

    matches = season_data.get("matches", {})
    all_events: list[dict[str, Any]] = []

    for match_id, match_data in matches.items():
        if match_data.get("status") != "success":
            continue

        match_url = match_data.get("match_url")
        events = match_data.get("events", [])

        for event in events:
            all_events.append({
                **event,
                "match_id": match_id,
                "match_url": match_url,
            })

    return pd.DataFrame.from_records(all_events)

In [12]:
def normalize_match_events_df(me_df: pd.DataFrame) -> pd.DataFrame:
    """Cleans and standardizes raw WhoScored match events."""
    df = me_df.copy()

    # Keep only events associated with a player.
    df.dropna(subset="playerId", inplace=True)

    df = df.where(pd.notnull(df), None)

    # Standardize column names.
    df = df.rename({
        "eventId": "event_id",
        "expandedMinute": "expanded_minute",
        "outcomeType": "outcome_type",
        "isTouch": "is_touch",
        "playerId": "player_id",
        "playerName": "player_name",
        "teamId": "team_id",
        "endX": "end_x",
        "endY": "end_y",
        "blockedX": "blocked_x",
        "blockedY": "blocked_y",
        "goalMouthZ": "goal_mouth_z",
        "goalMouthY": "goal_mouth_y",
        "isShot": "is_shot",
        "cardType": "card_type",
        "isGoal": "is_goal",
        "satisfiedEventsTypes": "satisfied_events_types",
        "relatedPlayerId": "related_player_id",
        "relatedPlayerName": "related_player_name",
        "relatedEventId": "related_event_id",
    }, axis=1)

    # Extract readable names from nested WhoScored fields.
    df["period_display_name"] = df["period"].apply(lambda x: x.get("displayName"))
    df["type_display_name"] = df["type"].apply(lambda x: x.get("displayName"))
    df["outcome_type_display_name"] = df["outcome_type"].apply(lambda x: x.get("displayName"))

    if "is_goal" not in df.columns:
        df["is_goal"] = False

    # Remove offside events from the analysis dataset.
    # df = df[~(df["type_display_name"] == "OffsideGiven")]

    df.drop(columns=["period", "type", "outcome_type"], inplace=True)

    df = df[[
        "id", "match_id", "event_id", "related_event_id",
        "minute", "second", "team_id", "player_id", "player_name",
        "related_player_id", "related_player_name", "match_url",
        "x", "y", "end_x", "end_y", "qualifiers", "is_touch",
        "blocked_x", "blocked_y", "goal_mouth_z", "goal_mouth_y",
        "is_shot", "card_type", "is_goal",
        "period_display_name", "type_display_name", "outcome_type_display_name",
    ]]

    # Convert core IDs and numeric fields.
    df[["id", "event_id", "minute", "team_id", "player_id"]] = (
        df[["id", "event_id", "minute", "team_id", "player_id"]].astype(int)
    )

    df[["second", "x", "y", "end_x", "end_y"]] = (
        df[["second", "x", "y", "end_x", "end_y"]].astype(float)
    )

    df["is_goal"] = df["is_goal"].fillna(False).astype(bool)
    df["is_shot"] = df["is_shot"].fillna(False).astype(bool)
    df["card_type"] = df["card_type"].fillna(False).astype(bool)

    return df

In [13]:
def _build_match_events_csv_output_path(
    country_name: str,
    league_name: str,
    season: str,
) -> Path:
    """Builds the CSV output path for cleaned match events."""
    safe_country = country_name.lower().replace(" ", "_")
    safe_league = league_name.lower().replace(" ", "_")
    safe_season = season.replace("/", "_")

    return Path("data/match_events") / f"{safe_country}_{safe_league}_{safe_season}.csv"


def save_match_events_df_to_csv(
    df: pd.DataFrame,
    country_name: str,
    league_name: str,
    season: str,
) -> Path:
    """Saves match events DataFrame as a CSV file."""
    output_path = _build_match_events_csv_output_path(
        country_name=country_name,
        league_name=league_name,
        season=season,
    )

    output_path.parent.mkdir(parents=True, exist_ok=True)

    df.to_csv(output_path, index=False, encoding="utf-8")

    print(f"Saved CSV: {output_path}")

    return output_path

In [14]:
SHOT_EVENTS = {
    "SavedShot",
    "MissedShots",
    "ShotOnPost",
    "Goal",
    "ChanceMissed",
}

DEFENSIVE_EVENTS = {
    "Tackle",
    "Interception",
    "Clearance",
    "BallRecovery",
    "Challenge",
    "OffsideProvoked",
}

GOALKEEPER_EVENTS = {
    "Save",
    "Claim",
    "Punch",
    "KeeperSweeper",
    "KeeperPickup",
    "Smother",
    "PenaltyFaced",
    "CrossNotClaimed",
}

def classify_event_group(row):
    event_type = row["type_display_name"]

    if event_type in SHOT_EVENTS or bool(row.get("is_shot", False)):
        return "Shot"

    if event_type in {"Pass", "BlockedPass", "OffsidePass"}:
        return "Pass"

    if event_type == "TakeOn":
        return "Dribble"

    if event_type in {"GoodSkill", "BallTouch", "ShieldBallOpp"}:
        return "Ball Action"

    if event_type in DEFENSIVE_EVENTS:
        return "Defensive Action"

    if event_type == "Aerial":
        return "Aerial Duel"

    if event_type in {"Dispossessed", "Error"}:
        return "Negative Action"

    if event_type in {"Foul", "Card"}:
        return "Foul / Discipline"

    if event_type in GOALKEEPER_EVENTS:
        return "Goalkeeper Action"

    if event_type == "CornerAwarded":
        return "Set Piece / Restart"

    if event_type in {"SubstitutionOff", "SubstitutionOn"}:
        return "Substitution"

    return "Other"

In [ ]:
# Step 1 — Load league teams and build team fixture URLs

league_teams = get_teams_by_league_from_whoscored(
    league_url=WHOSCORED_CONFIG.url,
)

league_fixture_urls = build_whoscored_fixture_urls(
    league_teams=league_teams,
    country_name=str(WHOSCORED_CONFIG.country_name).lower(),
)

In [ ]:
# Step 2 — Scrape league fixtures and store them

league_fixtures = scrape_league_fixtures(
    league_fixture_urls=league_fixture_urls,
    min_sleep=0.5,
    max_sleep=3,
)

if league_fixtures:
    fixtures_data = save_fixtures_to_json(
        fixtures=league_fixtures,
        country_name=WHOSCORED_CONFIG.country_name,
        league_name=WHOSCORED_CONFIG.league_name,
        season=WHOSCORED_CONFIG.season,
        output_path=FIXTURES_JSON_PATH,
    )

In [14]:
def _build_team_id_to_name_from_fixtures(
    fixtures: list[dict[str, Any]],
    league_name: str,
) -> dict[int, str]:
    """Builds a team ID-to-name map from stored league fixtures."""
    team_id_to_name: dict[int, str] = {}

    for fixture in fixtures:
        if fixture.get("competition") != league_name:
            continue

        home_team_id = fixture.get("home_team_id")
        home_team_name = fixture.get("home_team")
        away_team_id = fixture.get("away_team_id")
        away_team_name = fixture.get("away_team")

        if home_team_id and home_team_name:
            team_id_to_name[home_team_id] = home_team_name

        if away_team_id and away_team_name:
            team_id_to_name[away_team_id] = away_team_name

    return team_id_to_name

stored_fixtures = load_fixtures_from_json(
        country_name=WHOSCORED_CONFIG.country_name,
        league_name=WHOSCORED_CONFIG.league_name,
        season=WHOSCORED_CONFIG.season,
        file_path=FIXTURES_JSON_PATH,
    )

team_id_to_name = _build_team_id_to_name_from_fixtures(
    fixtures=stored_fixtures,
    league_name=WHOSCORED_CONFIG.league_name,
)

In [ ]:
# Step 3 — Extract played and not played match URLs

fixture_urls_by_status = extract_match_urls(
    fixtures=league_fixtures,
    country_name=WHOSCORED_CONFIG.country_name,
    league_name=WHOSCORED_CONFIG.league_name,
)

total_fixture_urls = (
    len(fixture_urls_by_status["fixtures_played"])
    + len(fixture_urls_by_status["fixtures_not_played"])
)

print(f"Successfully extracted: {total_fixture_urls} match URLs")

played_match_urls = fixture_urls_by_status["fixtures_played"]

print(
    f"Found {len(played_match_urls)} played "
    f"{WHOSCORED_CONFIG.league_name} fixtures."
)

In [ ]:
# Step 4 — Scrape match events from played fixtures

if played_match_urls:
    match_events_data = scrape_match_events_in_batches(
        match_urls=played_match_urls,
        country_name=WHOSCORED_CONFIG.country_name,
        league_name=WHOSCORED_CONFIG.league_name,
        season=WHOSCORED_CONFIG.season,
        output_path=MATCH_EVENTS_JSON_PATH,
        batch_size=10,
    )
else:
    print("No played match URLs found.")

In [ ]:
# Step 5 — Load, clean, and enrich match events

all_match_events_df = load_all_match_events_df(
    country_name=WHOSCORED_CONFIG.country_name,
    league_name=WHOSCORED_CONFIG.league_name,
    season=WHOSCORED_CONFIG.season,
    file_path=MATCH_EVENTS_JSON_PATH,
)

print(
    f"Total events loaded for "
    f"{WHOSCORED_CONFIG.country_name} - "
    f"{WHOSCORED_CONFIG.league_name} - "
    f"{WHOSCORED_CONFIG.season}: "
    f"{all_match_events_df.shape[0]}"
)

all_match_events_cleaned_df = normalize_match_events_df(
    me_df=all_match_events_df,
)

all_match_events_cleaned_df["team_name"] = (
    all_match_events_cleaned_df["team_id"].map(team_id_to_name)
)

all_match_events_cleaned_df["event_group"] = all_match_events_cleaned_df.apply(classify_event_group, axis=1)

In [ ]:
print(all_match_events_df.shape[0])
print(all_match_events_cleaned_df.shape[0])

In [ ]:
csv_output_path = save_match_events_df_to_csv(
    df=all_match_events_cleaned_df,
    country_name=WHOSCORED_CONFIG.country_name,
    league_name=WHOSCORED_CONFIG.league_name,
    season=WHOSCORED_CONFIG.season,
)

In [ ]:
df = pd.read_csv(str(csv_output_path), low_memory=False)